#Extract Data and Load in Bronze Layer

In [0]:
# edit
from pyspark.sql import DataFrame
from datetime import datetime
import traceback

# Source path
source_path = "/Volumes/workspace/bronze/source_systems/Raw Data/"

# Get all CSV files
files = dbutils.fs.ls(source_path)
csv_files = [f.name for f in files if f.name.endswith('.csv')]

print(f"Found {len(csv_files)} CSV files to process\n")

# Track results
results = []

for file_name in csv_files:
    table_name = file_name.replace('.csv', '').lower()
    file_path = source_path + file_name
    
    try:
        start_time = datetime.now()
        
        # Read CSV
        df = spark.read.option("header", "true").option("inferSchema", "true").csv(file_path)
        
        # Get row count
        row_count = df.count()
        col_count = len(df.columns)
        
        # Write to bronze layer
        df.write.mode('overwrite').saveAsTable(f'bronze.{table_name}')
        
        # Validate by reading back
        validation_df = spark.table(f'bronze.{table_name}')
        validation_count = validation_df.count()
        
        # Check if counts match
        status = "✓ SUCCESS" if row_count == validation_count else "✗ MISMATCH"
        
        end_time = datetime.now()
        duration = (end_time - start_time).total_seconds()
        
        results.append({
            'file': file_name,
            'table': f'bronze.{table_name}',
            'status': status,
            'rows': row_count,
            'columns': col_count,
            'validated_rows': validation_count,
            'duration_sec': round(duration, 2)
        })
        
        print(f"{status} | {table_name:30} | Rows: {row_count:>10,} | Cols: {col_count:>3} | {duration:.2f}s")
        
    except Exception as e:
        results.append({
            'file': file_name,
            'table': f'bronze.{table_name}',
            'status': '✗ FAILED',
            'rows': 0,
            'columns': 0,
            'validated_rows': 0,
            'duration_sec': 0,
            'error': str(e)
        })
        print(f"✗ FAILED | {table_name:30} | Error: {str(e)}")

# Summary
print("\n" + "="*80)
print("SUMMARY")
print("="*80)

success_count = len([r for r in results if r['status'] == '✓ SUCCESS'])
failed_count = len([r for r in results if r['status'] == '✗ FAILED'])
total_rows = sum([r['rows'] for r in results])

print(f"Total files processed: {len(results)}")
print(f"Successful loads: {success_count}")
print(f"Failed loads: {failed_count}")
print(f"Total rows loaded: {total_rows:,}")

if failed_count > 0:
    print("\nFailed tables:")
    for r in results:
        if r['status'] == '✗ FAILED':
            print(f"  - {r['table']}: {r.get('error', 'Unknown error')}")

# Create results DataFrame for easy viewing
results_df = spark.createDataFrame(results)
display(results_df)
